# Edge Interpretation — Stage 1.5 Walk-Forward Attribution

Attribution analyses over Stage 1.5 audit logs to answer **where the edge (if any) actually comes from**.
Built to be safe on partial Stage 1.5 outputs — every section degrades gracefully if data is missing.

**Inputs**
- Stage 1.5 audit log parquets: `polymarket/research/data/backtests/stage15/*.parquet`
- Stage 1.5 summary JSONs: `polymarket/research/data/backtests/stage15/*_summary.json`
- Markets snapshot: `polymarket/research/data/markets/markets_*.parquet`
- Trader / position / bankroll parquets (used in select sections only)

**Sections**
1. Category attribution
2. Temporal patterns
3. Leader concentration
4. Market characteristics
5. Cross-cohort overlap
6. Slippage forensics
7. Per-market exposure (concurrent)
8. Synthesis (the section that actually matters)

**Conventions**
- *Primary windows* = `2025`, `2026`. *Sidebar* = `2024` (low-confidence).
- A "combination" = `(cohort, resolution_bucket, sizing_rule, bt_window)`.
- Plots use matplotlib only (no seaborn).
- DuckDB SQL handles aggregation, pandas only at the edge for display.


In [ ]:
# --- Setup: imports, paths, DuckDB ---
from __future__ import annotations

import glob
import json
import os
import re
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Repo root — try a few candidates so the notebook is portable.
def _find_repo_root() -> Path:
    explicit = os.environ.get("EPSILON_REPO_ROOT")
    if explicit and (Path(explicit) / "polymarket" / "research").exists():
        return Path(explicit)
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "polymarket" / "research").exists():
            return parent
    # Last-ditch hardcoded fallback for this machine.
    return Path("/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research")

REPO_ROOT = _find_repo_root()

# Stage 1.5 outputs live at polymarket/research/data/backtests/stage15/ (the handoff path).
# Some specs use "stage_15" — accept either, and allow override via env var.
STAGE15_DIR = Path(os.environ.get(
    "STAGE15_DIR",
    str(REPO_ROOT / "polymarket/research" / "data" / "backtests" / "stage15"),
))
if not STAGE15_DIR.exists():
    alt = REPO_ROOT / "polymarket/research" / "data" / "backtests" / "stage_15"
    if alt.exists():
        STAGE15_DIR = alt

MARKETS_DIR = REPO_ROOT / "polymarket/research" / "data" / "markets"
TRADERS_PARQUET = REPO_ROOT / "polymarket/research" / "data" / "traders.parquet"
CLOSED_POS_PARQUET = REPO_ROOT / "polymarket/research" / "data" / "closed_positions.parquet"
BANKROLL_PARQUET = REPO_ROOT / "polymarket/research" / "data" / "bankroll_timeseries.parquet"

# Strategy capital assumption for percentage-of-capital math (Section 7).
# Stage 1.5 used fixed 1% sizing => $1000 copy => $100k notional.
STRATEGY_CAPITAL = float(os.environ.get("STRATEGY_CAPITAL", 100_000))

PRIMARY_WINDOWS = {"2025", "2026"}
SIDEBAR_WINDOWS = {"2024"}

print(f"REPO_ROOT       = {REPO_ROOT}")
print(f"STAGE15_DIR     = {STAGE15_DIR}")
print(f"MARKETS_DIR     = {MARKETS_DIR}")
print(f"STRATEGY_CAPITAL = ${STRATEGY_CAPITAL:,.0f}")

con = duckdb.connect()
con.execute("PRAGMA threads=4")


In [ ]:
# --- Load audit log union as a DuckDB view ---
AUDIT_GLOB = str(STAGE15_DIR / "*.parquet")
audit_files = [p for p in sorted(glob.glob(AUDIT_GLOB)) if not p.endswith("_summary.json")]
# Filter out any *_summary.parquet just in case (we expect summaries to be .json, but be defensive).
audit_files = [p for p in audit_files if "_summary" not in Path(p).stem]

# Latest markets snapshot (alphabetical sort works because filenames embed ISO dates).
market_files = sorted(MARKETS_DIR.glob("markets_*.parquet"))
MARKETS_PARQUET = str(market_files[-1]) if market_files else None

HAVE_AUDIT = len(audit_files) > 0

print(f"audit_files: {len(audit_files)} parquet(s) in {STAGE15_DIR}")
print(f"markets_parquet: {MARKETS_PARQUET}")

if not HAVE_AUDIT:
    print("\nStage 1.5 not yet complete — re-run when audit logs exist.")
else:
    files_sql = "[" + ",".join(f"'{p}'" for p in audit_files) + "]"
    con.execute(f"""
        CREATE OR REPLACE VIEW audit_raw AS
        SELECT * FROM read_parquet({files_sql})
    """)

    # Window is encoded in run_id: `{cohort}__{bucket}__{sizing}__{bt_window}__{suffix}`.
    # cohort and sizing may contain single underscores, but `__` is the splitter.
    con.execute("""
        CREATE OR REPLACE VIEW audit AS
        SELECT
            *,
            split_part(run_id, '__', 4) AS bt_window,
            CASE WHEN split_part(run_id, '__', 4) IN ('2025','2026') THEN TRUE ELSE FALSE END AS is_primary,
            -- combination key for grouping
            cohort || '|' || resolution_bucket || '|' || sizing_rule || '|' || split_part(run_id, '__', 4) AS combo
        FROM audit_raw
    """)

    # Markets view with derived category (no native category column).
    if MARKETS_PARQUET:
        con.execute(f"""
            CREATE OR REPLACE VIEW markets_raw AS
            SELECT * FROM read_parquet('{MARKETS_PARQUET}')
        """)
        con.execute("""
            CREATE OR REPLACE VIEW markets AS
            SELECT
                CAST(id AS VARCHAR) AS market_id,
                question,
                slug,
                volume,
                liquidity,
                end_date,
                neg_risk,
                CASE
                    WHEN regexp_matches(slug, '^(nba|mlb|nfl|nhl|mls|cfb|cbb|ncaa|epl|ucl|uefa|wnba|f1|nascar|ufc|mma|atp|wta)[-_]')
                        OR regexp_matches(lower(question), '(nba|mlb|nfl|nhl|premier league|stanley cup|super bowl|world series|champions league|formula 1|ufc|tennis|golf|pga)')
                        THEN 'sports'
                    WHEN regexp_matches(slug, '(bitcoin|btc|ethereum|eth|solana|sol|xrp|dogecoin|crypto|hyperliquid|fartcoin|altcoin|memecoin)')
                        OR regexp_matches(slug, '^(btc|eth|sol|xrp|doge)-')
                        THEN 'crypto'
                    WHEN regexp_matches(slug, '(election|senate|president|presidential|congress|governor|mayor|parliament|prime-minister|cabinet|gop|democrat|republican|biden|trump|harris|vance|kamala)')
                        THEN 'politics'
                    WHEN regexp_matches(slug, '(temperature|weather|hurricane|storm|earthquake|tornado)')
                        THEN 'weather'
                    WHEN regexp_matches(slug, '(fed|fomc|inflation|cpi|gdp|unemployment|jobs-report|interest-rate|rate-cut|rate-hike|recession)')
                        THEN 'macro'
                    WHEN regexp_matches(slug, '(spacex|nasa|launch|starship|rocket|space)')
                        THEN 'science_tech'
                    WHEN regexp_matches(slug, '(oscar|emmy|grammy|box-office|movie|film|album|song|chart|grammys|tony|netflix|disney)')
                        THEN 'entertainment'
                    WHEN regexp_matches(slug, '(tweet|elon|musk|x-account|posts)')
                        THEN 'social_media'
                    WHEN regexp_matches(slug, '(ai|openai|gpt|anthropic|claude|gemini|llm|chatgpt)')
                        THEN 'ai'
                    ELSE 'other'
                END AS category
            FROM markets_raw
        """)

        # Final analytic view: audit + category + market lifetime volume/liquidity.
        con.execute("""
            CREATE OR REPLACE VIEW audit_x AS
            SELECT
                a.*,
                COALESCE(m.category, 'other') AS category,
                m.question     AS market_question,
                m.slug         AS market_slug,
                m.volume       AS market_volume_lifetime,
                m.liquidity    AS market_liquidity_lifetime,
                m.end_date     AS market_end_date
            FROM audit a
            LEFT JOIN markets m USING (market_id)
        """)
    else:
        con.execute("""
            CREATE OR REPLACE VIEW audit_x AS
            SELECT
                a.*,
                'other'::VARCHAR AS category,
                NULL::VARCHAR    AS market_question,
                NULL::VARCHAR    AS market_slug,
                NULL::DOUBLE     AS market_volume_lifetime,
                NULL::DOUBLE     AS market_liquidity_lifetime,
                NULL::VARCHAR    AS market_end_date
            FROM audit a
        """)

    # Coverage stats.
    summary = con.execute("""
        SELECT
            COUNT(*) AS n_rows,
            MIN(trade_timestamp) AS dt_min,
            MAX(trade_timestamp) AS dt_max,
            COUNT(DISTINCT combo) AS n_combos,
            COUNT(DISTINCT cohort) AS n_cohorts,
            COUNT(DISTINCT bt_window) AS n_windows
        FROM audit_x
    """).df()
    print("\nAudit union coverage:")
    print(summary.to_string(index=False))

    print("\nCombinations available:")
    combos_df = con.execute("""
        SELECT cohort, resolution_bucket, sizing_rule, bt_window, COUNT(*) AS n
        FROM audit_x GROUP BY 1,2,3,4 ORDER BY 1,2,3,4
    """).df()
    print(f"  {len(combos_df)} combo rows; first 10:")
    print(combos_df.head(10).to_string(index=False))


def require_audit(section: str) -> bool:
    """Section guard: return True if audit data is loaded, else print 'Awaiting' and return False."""
    if not HAVE_AUDIT:
        print(f"[{section}] Awaiting Stage 1.5 — no audit parquets yet.")
        return False
    return True


def primary_only(combos_df: pd.DataFrame) -> pd.DataFrame:
    return combos_df[combos_df["bt_window"].isin(PRIMARY_WINDOWS)].copy()


## Section 1 — Category Attribution

For each primary `(cohort, bucket, sizing, bt_window)`:
- Per-category PnL, signals, win-rate, avg PnL/signal, monthly Sharpe.
- Stacked bar chart faceted by cohort.
- Tables: consistently-negative categories (denylist candidates) and specialist-favoured categories.


In [ ]:
# --- Section 1: Category Attribution ---
if require_audit("Section 1"):
    cat_combo = con.execute("""
        SELECT
            cohort, resolution_bucket, sizing_rule, bt_window, category,
            COUNT(*)                                     AS n_signals,
            SUM(copy_pnl_usd)                            AS sum_pnl,
            AVG(copy_pnl_usd)                            AS avg_pnl_per_signal,
            AVG(CASE WHEN copy_pnl_usd > 0 THEN 1.0 ELSE 0.0 END) AS win_rate,
            AVG(slippage_cents)                          AS avg_slip_cents
        FROM audit_x
        WHERE bt_window IN ('2025','2026')
        GROUP BY 1,2,3,4,5
        ORDER BY 1,2,3,4, sum_pnl DESC
    """).df()

    # Monthly Sharpe per (combo, category): mean(monthly_pnl) / std(monthly_pnl) * sqrt(12).
    monthly = con.execute("""
        WITH per_month AS (
            SELECT
                cohort, resolution_bucket, sizing_rule, bt_window, category,
                date_trunc('month', trade_timestamp) AS ym,
                SUM(copy_pnl_usd) AS month_pnl
            FROM audit_x
            WHERE bt_window IN ('2025','2026')
            GROUP BY 1,2,3,4,5,6
        )
        SELECT
            cohort, resolution_bucket, sizing_rule, bt_window, category,
            AVG(month_pnl) AS mean_month_pnl,
            STDDEV_SAMP(month_pnl) AS sd_month_pnl,
            COUNT(*) AS n_months
        FROM per_month
        GROUP BY 1,2,3,4,5
    """).df()
    monthly["monthly_sharpe"] = np.where(
        (monthly["sd_month_pnl"].fillna(0) > 0),
        (monthly["mean_month_pnl"] / monthly["sd_month_pnl"]) * np.sqrt(12),
        np.nan,
    )

    cat_combo = cat_combo.merge(
        monthly[["cohort", "resolution_bucket", "sizing_rule", "bt_window", "category", "monthly_sharpe", "n_months"]],
        on=["cohort", "resolution_bucket", "sizing_rule", "bt_window", "category"],
        how="left",
    )

    if cat_combo.empty:
        print("Section 1: no primary-bt_window (2025/2026) audit rows yet.")
    else:
        print(f"Section 1: per-category × per-combo rows = {len(cat_combo)}")
        # ----- Stacked bar chart per cohort -----
        cohorts = cat_combo["cohort"].unique()
        n = len(cohorts)
        fig, axes = plt.subplots(1, max(n, 1), figsize=(6 * max(n, 1), 4.2), squeeze=False)
        for ax, ch in zip(axes[0], cohorts):
            sub = (
                cat_combo[cat_combo["cohort"] == ch]
                .groupby("category")["sum_pnl"]
                .sum()
                .sort_values()
            )
            colors = ["#c0392b" if v < 0 else "#27ae60" for v in sub.values]
            ax.barh(sub.index, sub.values, color=colors)
            ax.axvline(0, color="black", lw=0.6)
            ax.set_title(f"{ch} — $ PnL by category")
            ax.set_xlabel("PnL ($)")
        plt.tight_layout(); plt.show()

        # ----- Consistently-negative categories (across multiple cohorts) -----
        denylist_candidates = (
            cat_combo.groupby(["category", "cohort"])["sum_pnl"]
            .sum()
            .reset_index()
        )
        denylist_summary = (
            denylist_candidates.groupby("category")
            .agg(
                cohorts_negative=("sum_pnl", lambda s: int((s < 0).sum())),
                cohorts_total=("sum_pnl", "count"),
                total_pnl=("sum_pnl", "sum"),
            )
            .reset_index()
            .sort_values(["cohorts_negative", "total_pnl"], ascending=[False, True])
        )
        denylist_summary = denylist_summary[
            (denylist_summary["cohorts_negative"] >= 2) & (denylist_summary["total_pnl"] < 0)
        ]
        print("\nDenylist candidates (negative across ≥2 cohorts):")
        print(denylist_summary.to_string(index=False) if len(denylist_summary) else "  (none)")

        # ----- Specialist-favoured categories: one cohort dominates -----
        specialist = (
            cat_combo.groupby(["category", "cohort"])["sum_pnl"]
            .sum()
            .reset_index()
        )
        def _spec_row(grp):
            grp = grp.sort_values("sum_pnl", ascending=False)
            total_pos = grp.loc[grp["sum_pnl"] > 0, "sum_pnl"].sum()
            top_pnl = grp.iloc[0]["sum_pnl"]
            top_cohort = grp.iloc[0]["cohort"]
            share = (top_pnl / total_pos) if total_pos > 0 else np.nan
            return pd.Series({"top_cohort": top_cohort, "top_pnl": top_pnl,
                              "share_of_positive": share, "n_cohorts_positive": int((grp["sum_pnl"] > 0).sum())})
        spec_df = specialist.groupby("category").apply(_spec_row).reset_index()
        spec_df = spec_df[(spec_df["share_of_positive"] >= 0.66) & (spec_df["top_pnl"] > 0)].sort_values("top_pnl", ascending=False)
        print("\nSpecialist categories (one cohort owns ≥66% of positive PnL):")
        print(spec_df.to_string(index=False) if len(spec_df) else "  (none)")


## Section 2 — Temporal Patterns

- Top-5 primary combinations by **monthly Sharpe**.
- Cumulative PnL curves.
- Monthly PnL boxplots per cohort.
- Concentration check: months > 25% of total PnL.
- Rolling 3-month Sharpe.
- Top single-month contributors, with light-touch event annotation.


In [ ]:
# --- Section 2: Temporal Patterns ---
EVENT_HINTS = {
    ("2024-11", "2024-12"): "US presidential election cycle",
    ("2024-07", "2024-08"): "Trump RNC / Biden withdrawal",
    ("2025-01", "2025-01"): "US inauguration",
    ("2025-11", "2025-12"): "Off-cycle elections / end-of-year",
}

def _event_hint(ym: pd.Timestamp) -> str:
    key = ym.strftime("%Y-%m")
    for (lo, hi), label in EVENT_HINTS.items():
        if lo <= key <= hi:
            return label
    return ""


if require_audit("Section 2"):
    # Sharpe per combo from monthly PnL.
    combo_monthly = con.execute("""
        SELECT cohort, resolution_bucket, sizing_rule, bt_window,
               date_trunc('month', trade_timestamp) AS ym,
               SUM(copy_pnl_usd) AS month_pnl,
               SUM(copy_size_usd) AS month_size
        FROM audit_x
        WHERE bt_window IN ('2025','2026')
        GROUP BY 1,2,3,4,5
        ORDER BY 1,2,3,4,5
    """).df()

    if combo_monthly.empty:
        print("Section 2: no primary-bt_window audit rows yet.")
    else:
        combo_keys = ["cohort", "resolution_bucket", "sizing_rule", "bt_window"]
        sharpe_df = (
            combo_monthly.groupby(combo_keys)["month_pnl"]
            .agg(["mean", "std", "count", "sum"])
            .reset_index()
        )
        sharpe_df["monthly_sharpe"] = np.where(
            sharpe_df["std"].fillna(0) > 0,
            (sharpe_df["mean"] / sharpe_df["std"]) * np.sqrt(12),
            np.nan,
        )
        sharpe_df = sharpe_df.dropna(subset=["monthly_sharpe"]).sort_values("monthly_sharpe", ascending=False)
        top5 = sharpe_df.head(5).reset_index(drop=True)
        print("Top-5 primary combinations by monthly Sharpe:")
        print(top5[combo_keys + ["count", "sum", "monthly_sharpe"]].to_string(index=False))

        if top5.empty:
            print("  (no Sharpe-rankable combinations — need ≥2 months of PnL)")
        else:
            # ----- Cumulative PnL curves -----
            fig, ax = plt.subplots(figsize=(11, 5))
            for _, r in top5.iterrows():
                sub = combo_monthly.merge(pd.DataFrame([r[combo_keys]]), on=combo_keys, how="inner").sort_values("ym")
                sub["cum_pnl"] = sub["month_pnl"].cumsum()
                label = " | ".join(str(r[c]) for c in combo_keys)
                ax.plot(sub["ym"], sub["cum_pnl"], marker="o", label=label, lw=1.4, ms=4)
            ax.axhline(0, color="black", lw=0.6)
            ax.set_title("Cumulative PnL — top-5 primary combinations")
            ax.set_ylabel("Cumulative PnL ($)")
            ax.legend(loc="best", fontsize=7)
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            fig.autofmt_xdate(); plt.tight_layout(); plt.show()

            # ----- Monthly PnL boxplot per cohort -----
            primary_monthly = combo_monthly.copy()
            cohorts = sorted(primary_monthly["cohort"].unique())
            if cohorts:
                fig, ax = plt.subplots(figsize=(9, 4.5))
                data = [primary_monthly.loc[primary_monthly["cohort"] == c, "month_pnl"].values for c in cohorts]
                ax.boxplot(data, tick_labels=cohorts, showmeans=True)
                ax.axhline(0, color="black", lw=0.6)
                ax.set_title("Monthly PnL distribution per cohort (primary windows)")
                ax.set_ylabel("Monthly PnL ($)")
                plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

            # ----- Concentration: months > 25% of total PnL for top-5 combos -----
            concentration_rows = []
            for _, r in top5.iterrows():
                sub = combo_monthly.merge(pd.DataFrame([r[combo_keys]]), on=combo_keys, how="inner").sort_values("ym")
                total = sub["month_pnl"].sum()
                if total <= 0:
                    continue
                sub["share"] = sub["month_pnl"] / total
                hot = sub[sub["share"] > 0.25]
                for _, h in hot.iterrows():
                    concentration_rows.append({
                        **{c: r[c] for c in combo_keys},
                        "ym": h["ym"].strftime("%Y-%m"),
                        "month_pnl": h["month_pnl"],
                        "share_of_total": h["share"],
                        "event_hint": _event_hint(h["ym"]),
                    })
            conc_df = pd.DataFrame(concentration_rows)
            print("\nMonths contributing >25% of total PnL for a top-5 combo:")
            print(conc_df.to_string(index=False) if len(conc_df) else "  (none)")

            # ----- Rolling 3-month Sharpe -----
            fig, ax = plt.subplots(figsize=(11, 4.5))
            for _, r in top5.iterrows():
                sub = combo_monthly.merge(pd.DataFrame([r[combo_keys]]), on=combo_keys, how="inner").sort_values("ym").reset_index(drop=True)
                if len(sub) < 3:
                    continue
                roll_mean = sub["month_pnl"].rolling(3).mean()
                roll_std = sub["month_pnl"].rolling(3).std()
                roll_sharpe = (roll_mean / roll_std) * np.sqrt(12)
                label = " | ".join(str(r[c]) for c in combo_keys)
                ax.plot(sub["ym"], roll_sharpe, marker=".", lw=1.2, label=label)
            ax.axhline(0, color="black", lw=0.6)
            ax.set_title("Rolling 3-month Sharpe — top-5 primary combinations")
            ax.set_ylabel("Sharpe (ann.)")
            ax.legend(loc="best", fontsize=7)
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            fig.autofmt_xdate(); plt.tight_layout(); plt.show()

            # ----- Largest single-month PnL contributors with top markets -----
            big_months = (
                combo_monthly.sort_values("month_pnl", ascending=False).head(10).copy()
            )
            top_market_rows = []
            for _, r in big_months.iterrows():
                ym = r["ym"]
                df_m = con.execute(f"""
                    SELECT market_id, ANY_VALUE(market_question) AS question,
                           ANY_VALUE(category) AS category,
                           SUM(copy_pnl_usd) AS pnl, COUNT(*) AS n
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                      AND date_trunc('month', trade_timestamp) = TIMESTAMP '{ym.strftime("%Y-%m-%d")}'
                    GROUP BY market_id
                    ORDER BY pnl DESC LIMIT 2
                """).df()
                for _, mrow in df_m.iterrows():
                    top_market_rows.append({
                        "combo": " | ".join(str(r[c]) for c in combo_keys),
                        "ym": ym.strftime("%Y-%m"),
                        "month_pnl": r["month_pnl"],
                        "market_id": mrow["market_id"],
                        "category": mrow["category"],
                        "question": (mrow["question"] or "")[:80],
                        "market_pnl": mrow["pnl"],
                        "event_hint": _event_hint(ym),
                    })
            print("\nLargest single-month contributors and their top-2 markets:")
            print(pd.DataFrame(top_market_rows).to_string(index=False) if top_market_rows else "  (none)")


## Section 3 — Leader Concentration

For each top combination (top-5 by monthly Sharpe):
- Per-leader PnL, refresh selections, signals, win-rate.
- Top 10 leaders by PnL contribution + bottom 5.
- Pareto curve of edge concentration.
- Durability scatter: # times selected × PnL/selection.
- Ranker-noise leaders: selected ≥3 times yet net-negative.


In [ ]:
# --- Section 3: Leader Concentration ---
if require_audit("Section 3"):
    # Reuse top-5 logic but recompute locally to keep cell standalone.
    s3_monthly = con.execute("""
        SELECT cohort, resolution_bucket, sizing_rule, bt_window,
               date_trunc('month', trade_timestamp) AS ym,
               SUM(copy_pnl_usd) AS month_pnl
        FROM audit_x WHERE bt_window IN ('2025','2026')
        GROUP BY 1,2,3,4,5
    """).df()

    if s3_monthly.empty:
        print("Section 3: no primary-bt_window audit rows yet.")
    else:
        keys = ["cohort", "resolution_bucket", "sizing_rule", "bt_window"]
        s3_sharpe = (
            s3_monthly.groupby(keys)["month_pnl"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        s3_sharpe["sharpe"] = np.where(s3_sharpe["std"].fillna(0) > 0,
                                       (s3_sharpe["mean"] / s3_sharpe["std"]) * np.sqrt(12), np.nan)
        s3_top = s3_sharpe.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False).head(5)

        if s3_top.empty:
            print("Section 3: not enough months/combos to rank.")
        else:
            ranker_noise_all = []
            for _, r in s3_top.iterrows():
                label = " | ".join(str(r[c]) for c in keys)
                print(f"\n=== {label} ===")
                ld = con.execute(f"""
                    SELECT leader_address,
                           SUM(copy_pnl_usd)  AS pnl_total,
                           COUNT(DISTINCT refresh_date) AS n_refresh_selections,
                           COUNT(*)           AS n_signals,
                           AVG(CASE WHEN copy_pnl_usd > 0 THEN 1.0 ELSE 0.0 END) AS win_rate
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                    GROUP BY leader_address
                """).df().sort_values("pnl_total", ascending=False)

                if ld.empty:
                    print("  no signals"); continue
                print(f"Top 10 leaders by PnL contribution (of {len(ld)}):")
                print(ld.head(10).to_string(index=False))
                print("\nBottom 5:")
                print(ld.tail(5).to_string(index=False))

                # ---- Pareto ----
                ld_sorted = ld.sort_values("pnl_total", ascending=False).reset_index(drop=True)
                pos_pnl_total = ld_sorted.loc[ld_sorted["pnl_total"] > 0, "pnl_total"].sum()
                fig, ax = plt.subplots(figsize=(8, 4))
                if pos_pnl_total > 0:
                    cum = ld_sorted["pnl_total"].clip(lower=0).cumsum() / pos_pnl_total
                    ax.plot(np.arange(1, len(cum) + 1) / len(cum), cum, marker=".", lw=1.2)
                    ax.plot([0, 1], [0, 1], "k--", lw=0.6, label="uniform")
                    ax.set_xlabel("Cumulative % of leaders")
                    ax.set_ylabel("Cumulative % of positive PnL")
                    ax.set_title(f"Pareto — {label}")
                    ax.legend(loc="lower right")
                else:
                    ax.text(0.5, 0.5, "no positive PnL", ha="center", va="center", transform=ax.transAxes)
                    ax.set_title(f"Pareto — {label}")
                plt.tight_layout(); plt.show()

                # ---- Durability scatter ----
                fig, ax = plt.subplots(figsize=(7, 4))
                if len(ld) > 0:
                    pnl_per_sel = np.where(ld["n_refresh_selections"] > 0,
                                           ld["pnl_total"] / ld["n_refresh_selections"], 0)
                    ax.scatter(ld["n_refresh_selections"], pnl_per_sel, alpha=0.5, s=24)
                    ax.axhline(0, color="black", lw=0.6)
                    ax.set_xlabel("# times selected (distinct refresh_dates)")
                    ax.set_ylabel("PnL per selection ($)")
                    ax.set_title(f"Durability — {label}")
                plt.tight_layout(); plt.show()

                # ---- Ranker-noise leaders ----
                rn = ld[(ld["n_refresh_selections"] >= 3) & (ld["pnl_total"] < 0)].copy()
                rn["combo"] = label
                ranker_noise_all.append(rn)

            if ranker_noise_all:
                ranker_noise = pd.concat(ranker_noise_all, ignore_index=True)
                print("\nRanker-noise leaders (selected ≥3 times, cumulative-negative):")
                print(ranker_noise.sort_values("pnl_total").to_string(index=False))
            else:
                print("\nNo ranker-noise leaders flagged.")


## Section 4 — Market Characteristics

For each top combination, slice by:
- Lifetime market volume buckets: `[$50k-$100k)`, `[$100k-$500k)`, `[$500k+)`.
- Days-to-resolution sub-buckets *inside* the resolution bucket (e.g. for 30d: 0-7, 7-14, 14-30).
- NegRisk vs regular.

Output: which market profile delivers the best PnL/signal.


In [ ]:
# --- Section 4: Market Characteristics ---
if require_audit("Section 4"):
    s4_monthly = con.execute("""
        SELECT cohort, resolution_bucket, sizing_rule, bt_window,
               date_trunc('month', trade_timestamp) AS ym,
               SUM(copy_pnl_usd) AS month_pnl
        FROM audit_x WHERE bt_window IN ('2025','2026')
        GROUP BY 1,2,3,4,5
    """).df()
    if s4_monthly.empty:
        print("Section 4: no primary-bt_window data.")
    else:
        keys = ["cohort", "resolution_bucket", "sizing_rule", "bt_window"]
        s4 = (s4_monthly.groupby(keys)["month_pnl"].agg(["mean","std","count"]).reset_index())
        s4["sharpe"] = np.where(s4["std"].fillna(0)>0, (s4["mean"]/s4["std"])*np.sqrt(12), np.nan)
        top = s4.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False).head(5)
        if top.empty:
            print("Section 4: not enough data to rank top combinations.");
        else:
            for _, r in top.iterrows():
                label = " | ".join(str(r[c]) for c in keys)
                print(f"\n=== {label} ===")

                vol_q = f"""
                    SELECT
                        CASE
                            WHEN market_volume_lifetime IS NULL THEN 'unknown'
                            WHEN market_volume_lifetime < 100000 THEN '$50k-$100k'
                            WHEN market_volume_lifetime < 500000 THEN '$100k-$500k'
                            ELSE '$500k+'
                        END AS vol_bucket,
                        COUNT(*) AS n,
                        SUM(copy_pnl_usd) AS total_pnl,
                        AVG(copy_pnl_usd) AS pnl_per_signal,
                        AVG(CASE WHEN copy_pnl_usd>0 THEN 1.0 ELSE 0.0 END) AS win_rate
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                    GROUP BY vol_bucket ORDER BY vol_bucket
                """
                print("By lifetime market volume:")
                print(con.execute(vol_q).df().to_string(index=False))

                # Days-to-resolution sub-buckets inside the bucket.
                bucket = r["resolution_bucket"]
                if bucket == "2d":
                    sub_case = "CASE WHEN days_to_resolution<=0 THEN '0d' WHEN days_to_resolution<=1 THEN '1d' ELSE '2d' END"
                elif bucket == "7d":
                    sub_case = "CASE WHEN days_to_resolution<=2 THEN '0-2d' WHEN days_to_resolution<=4 THEN '3-4d' ELSE '5-7d' END"
                elif bucket == "30d":
                    sub_case = "CASE WHEN days_to_resolution<=7 THEN '0-7d' WHEN days_to_resolution<=14 THEN '8-14d' ELSE '15-30d' END"
                else:
                    sub_case = "CASE WHEN days_to_resolution<=14 THEN '0-14d' WHEN days_to_resolution<=30 THEN '15-30d' ELSE '30d+' END"

                d2r_q = f"""
                    SELECT {sub_case} AS d2r_subbucket,
                           COUNT(*) AS n,
                           SUM(copy_pnl_usd) AS total_pnl,
                           AVG(copy_pnl_usd) AS pnl_per_signal,
                           AVG(CASE WHEN copy_pnl_usd>0 THEN 1.0 ELSE 0.0 END) AS win_rate
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                    GROUP BY d2r_subbucket ORDER BY d2r_subbucket
                """
                print("\nBy days-to-resolution sub-bucket:")
                print(con.execute(d2r_q).df().to_string(index=False))

                negrisk_q = f"""
                    SELECT neg_risk,
                           COUNT(*) AS n,
                           SUM(copy_pnl_usd) AS total_pnl,
                           AVG(copy_pnl_usd) AS pnl_per_signal,
                           AVG(CASE WHEN copy_pnl_usd>0 THEN 1.0 ELSE 0.0 END) AS win_rate
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                    GROUP BY neg_risk
                """
                print("\nBy NegRisk:")
                print(con.execute(negrisk_q).df().to_string(index=False))

            # Cross-combo winner: which (vol_bucket, d2r_subbucket, neg_risk) has the best PnL/signal overall?
            winner = con.execute("""
                SELECT
                    CASE
                        WHEN market_volume_lifetime IS NULL THEN 'unknown'
                        WHEN market_volume_lifetime < 100000 THEN '$50k-$100k'
                        WHEN market_volume_lifetime < 500000 THEN '$100k-$500k'
                        ELSE '$500k+'
                    END AS vol_bucket,
                    neg_risk,
                    COUNT(*) AS n,
                    AVG(copy_pnl_usd) AS pnl_per_signal,
                    SUM(copy_pnl_usd) AS total_pnl
                FROM audit_x
                WHERE bt_window IN ('2025','2026')
                GROUP BY vol_bucket, neg_risk
                HAVING n >= 25
                ORDER BY pnl_per_signal DESC
            """).df()
            print("\nBest market profile by PnL/signal (across primary combos, n>=25):")
            print(winner.to_string(index=False))


## Section 5 — Cross-Cohort Overlap

- Markets selected by 2+ cohorts within the same `(bt_window, resolution_bucket)`.
- Direction agreement vs disagreement.
- PnL on agree-vs-disagree subsets.
- Back-of-envelope "only-if-agreed" PnL/signal-count.


In [ ]:
# --- Section 5: Cross-Cohort Overlap ---
if require_audit("Section 5"):
    overlap = con.execute("""
        SELECT
            bt_window, resolution_bucket, market_id,
            COUNT(DISTINCT cohort) AS n_cohorts,
            COUNT(DISTINCT leader_direction) AS n_directions,
            SUM(copy_pnl_usd) AS pnl_sum,
            COUNT(*) AS n_signals
        FROM audit_x
        WHERE bt_window IN ('2025','2026')
        GROUP BY bt_window, resolution_bucket, market_id
    """).df()
    if overlap.empty:
        print("Section 5: no primary data.")
    else:
        multi = overlap[overlap["n_cohorts"] >= 2].copy()
        multi["agreed"] = multi["n_directions"] == 1
        print(f"Markets selected by ≥2 cohorts (same bt_window+bucket): {len(multi)} of {len(overlap)}")
        print("\nAgreement breakdown:")
        breakdown = (multi.groupby("agreed")
                          .agg(n_markets=("market_id","count"),
                               total_signals=("n_signals","sum"),
                               total_pnl=("pnl_sum","sum"),
                               pnl_per_signal=("pnl_sum", lambda s: s.sum()/max(multi.loc[s.index,"n_signals"].sum(),1)))
                          .reset_index())
        print(breakdown.to_string(index=False))

        # Hypothetical: only-if-agreed across multi-cohort markets
        agreed_keys = set(zip(multi.loc[multi["agreed"], "bt_window"],
                              multi.loc[multi["agreed"], "resolution_bucket"],
                              multi.loc[multi["agreed"], "market_id"]))
        if agreed_keys:
            keys_df = pd.DataFrame(list(agreed_keys), columns=["bt_window","resolution_bucket","market_id"])
            con.register("agreed_keys", keys_df)
            agreed_only = con.execute("""
                SELECT cohort, resolution_bucket, sizing_rule, bt_window,
                       COUNT(*) AS n_signals, SUM(copy_pnl_usd) AS pnl
                FROM audit_x a
                JOIN agreed_keys k USING (bt_window, resolution_bucket, market_id)
                WHERE bt_window IN ('2025','2026')
                GROUP BY cohort, resolution_bucket, sizing_rule, bt_window
                ORDER BY pnl DESC
            """).df()
            con.unregister("agreed_keys")
            print("\nIf we'd ONLY copied multi-cohort agreed-upon markets — per combo:")
            print(agreed_only.to_string(index=False))
        else:
            print("\nNo agreed-upon multi-cohort markets to evaluate.")


## Section 6 — Slippage Forensics

- Distribution of `slippage_source` per combination.
- Avg slippage_cents by source.
- PnL contribution by source.
- Heatmap (cohort × category) of avg slippage.
- Flag combinations with **fallback_pct > 30%** — slippage model is least reliable there.


In [ ]:
# --- Section 6: Slippage Forensics ---
if require_audit("Section 6"):
    src_dist = con.execute("""
        SELECT cohort, resolution_bucket, sizing_rule, bt_window, slippage_source,
               COUNT(*) AS n, AVG(slippage_cents) AS avg_slip,
               SUM(copy_pnl_usd) AS pnl
        FROM audit_x
        WHERE bt_window IN ('2024','2025','2026')
        GROUP BY 1,2,3,4,5
        ORDER BY 1,2,3,4,5
    """).df()
    if src_dist.empty:
        print("Section 6: no audit rows.");
    else:
        # Per-combo fallback %.
        per_combo = con.execute("""
            SELECT cohort, resolution_bucket, sizing_rule, bt_window,
                   COUNT(*) AS n_total,
                   AVG(slippage_cents) AS avg_slip,
                   SUM(CASE WHEN slippage_source='fallback' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS fallback_pct,
                   SUM(copy_pnl_usd) AS pnl
            FROM audit_x GROUP BY 1,2,3,4
            ORDER BY fallback_pct DESC
        """).df()
        print("Per-combo fallback %:")
        print(per_combo.head(20).to_string(index=False))

        high_fb = per_combo[per_combo["fallback_pct"] > 0.30]
        print(f"\nCombinations with fallback_pct > 30% (slippage model least reliable): {len(high_fb)}")
        if len(high_fb):
            print(high_fb.to_string(index=False))

        # Avg slippage by source.
        by_source = con.execute("""
            SELECT slippage_source, COUNT(*) AS n,
                   AVG(slippage_cents) AS avg_slip,
                   SUM(copy_pnl_usd) AS pnl
            FROM audit_x GROUP BY slippage_source
        """).df()
        print("\nAvg slippage by source (all combos):")
        print(by_source.to_string(index=False))

        # Heatmap cohort × category of avg slippage.
        heat = con.execute("""
            SELECT cohort, category, AVG(slippage_cents) AS avg_slip
            FROM audit_x WHERE bt_window IN ('2025','2026')
            GROUP BY 1,2
        """).df()
        if not heat.empty:
            pivot = heat.pivot(index="cohort", columns="category", values="avg_slip").fillna(np.nan)
            fig, ax = plt.subplots(figsize=(1 + 0.7 * len(pivot.columns), 0.5 + 0.6 * len(pivot.index) + 1))
            im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
            ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
            ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
            ax.set_title("Avg slippage (¢) by cohort × category (primary windows)")
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    v = pivot.values[i, j]
                    if not np.isnan(v):
                        ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                                color="white" if v < np.nanmean(pivot.values) else "black", fontsize=8)
            fig.colorbar(im, ax=ax, label="avg slippage ¢")
            plt.tight_layout(); plt.show()


## Section 7 — Per-Market Exposure ("market max")

For each top combination, compute concurrent open exposure per `market_id` over time using SQL bt_window logic
(position is open from `trade_timestamp` to `resolution_date`).

- Distribution: % of markets where peak concurrent exposure exceeded 3% / 5% / 8% / 10% of strategy capital.
- Hypothetical 5% per-market cap: how many trades would have been blocked, and what's the PnL impact?
- Recommended per-market cap.


In [ ]:
# --- Section 7: Per-Market Exposure ---
if require_audit("Section 7"):
    keys = ["cohort", "resolution_bucket", "sizing_rule", "bt_window"]

    s7_monthly = con.execute("""
        SELECT cohort, resolution_bucket, sizing_rule, bt_window,
               date_trunc('month', trade_timestamp) AS ym, SUM(copy_pnl_usd) AS month_pnl
        FROM audit_x WHERE bt_window IN ('2025','2026')
        GROUP BY 1,2,3,4,5
    """).df()
    if s7_monthly.empty:
        print("Section 7: no primary data.")
    else:
        s7 = (s7_monthly.groupby(keys)["month_pnl"].agg(["mean","std","count"]).reset_index())
        s7["sharpe"] = np.where(s7["std"].fillna(0)>0, (s7["mean"]/s7["std"])*np.sqrt(12), np.nan)
        top = s7.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False).head(5)

        all_dist = []
        for _, r in top.iterrows():
            label = " | ".join(str(r[c]) for c in keys)
            # Concurrent exposure per market: for each (combo, market_id), for each trade timestamp,
            # sum copy_size_usd of trades whose [trade_timestamp, resolution_date) interval contains that timestamp.
            q = f"""
                WITH combo_rows AS (
                    SELECT market_id, trade_timestamp, resolution_date, copy_size_usd, copy_pnl_usd
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                ),
                pairs AS (
                    SELECT a.market_id, a.trade_timestamp AS event_t, SUM(b.copy_size_usd) AS exposure_at_t
                    FROM combo_rows a
                    JOIN combo_rows b
                      ON a.market_id = b.market_id
                     AND b.trade_timestamp <= a.trade_timestamp
                     AND b.resolution_date  > a.trade_timestamp
                    GROUP BY a.market_id, a.trade_timestamp
                )
                SELECT market_id, MAX(exposure_at_t) AS peak_exposure_usd
                FROM pairs GROUP BY market_id
            """
            per_market = con.execute(q).df()
            if per_market.empty:
                continue
            per_market["peak_pct_capital"] = per_market["peak_exposure_usd"] / STRATEGY_CAPITAL
            print(f"\n=== {label} ===")
            print(f"markets: {len(per_market)}, "
                  f"max peak: ${per_market['peak_exposure_usd'].max():,.0f} "
                  f"({per_market['peak_pct_capital'].max()*100:.2f}% of capital)")
            for thresh in [0.03, 0.05, 0.08, 0.10]:
                share = (per_market["peak_pct_capital"] > thresh).mean()
                print(f"  share of markets with peak > {thresh*100:.0f}% of capital: {share*100:.1f}%")

            # Hypothetical 5% cap PnL impact.
            cap_usd = 0.05 * STRATEGY_CAPITAL
            cap_q = f"""
                WITH combo_rows AS (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY market_id ORDER BY trade_timestamp) AS rn
                    FROM audit_x
                    WHERE cohort='{r['cohort']}' AND resolution_bucket='{r['resolution_bucket']}'
                      AND sizing_rule='{r['sizing_rule']}' AND bt_window='{r['bt_window']}'
                ),
                cum AS (
                    SELECT
                        x.market_id,
                        x.trade_timestamp,
                        x.copy_size_usd,
                        x.copy_pnl_usd,
                        COALESCE(SUM(y.copy_size_usd), 0) AS prior_open_exposure
                    FROM combo_rows x
                    LEFT JOIN combo_rows y
                      ON x.market_id = y.market_id
                     AND y.trade_timestamp < x.trade_timestamp
                     AND y.resolution_date  > x.trade_timestamp
                    GROUP BY x.market_id, x.trade_timestamp, x.copy_size_usd, x.copy_pnl_usd
                )
                SELECT
                    SUM(CASE WHEN prior_open_exposure + copy_size_usd > {cap_usd} THEN 1 ELSE 0 END) AS blocked,
                    SUM(CASE WHEN prior_open_exposure + copy_size_usd > {cap_usd} THEN copy_pnl_usd ELSE 0 END) AS blocked_pnl,
                    COUNT(*) AS total_signals,
                    SUM(copy_pnl_usd) AS total_pnl
                FROM cum
            """
            cap = con.execute(cap_q).df().iloc[0]
            blocked_pct = (cap["blocked"] / cap["total_signals"]) if cap["total_signals"] else 0
            print(f"  hypothetical 5% per-market cap: "
                  f"{int(cap['blocked'])}/{int(cap['total_signals'])} ({blocked_pct*100:.1f}%) trades blocked, "
                  f"blocked PnL = ${cap['blocked_pnl']:,.0f} "
                  f"(total PnL = ${cap['total_pnl']:,.0f})")

            per_market["combo"] = label
            all_dist.append(per_market)

        if all_dist:
            allm = pd.concat(all_dist, ignore_index=True)
            # Recommend a cap: 95th percentile of peak exposures across top combos, rounded up to nearest 1%.
            p95 = allm["peak_pct_capital"].quantile(0.95)
            rec = max(0.03, np.ceil(p95 * 100) / 100)
            print(f"\nRecommended per-market cap (95th pctile peak across top combos, floor 3%): {rec*100:.0f}% of capital")

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(allm["peak_pct_capital"] * 100, bins=40, color="#3498db", edgecolor="white")
            for thresh, c in [(3,"#aaa"), (5,"#e67e22"), (8,"#c0392b"), (10,"#7f0000")]:
                ax.axvline(thresh, color=c, lw=1, ls="--", label=f"{thresh}%")
            ax.set_xlabel("Peak concurrent exposure per market (% of strategy capital)")
            ax.set_ylabel("# markets")
            ax.set_title("Per-market peak exposure distribution (top combos)")
            ax.legend()
            plt.tight_layout(); plt.show()


## Synthesis — what to change in the next strategy iteration

This cell consolidates section outputs into a punch-list. It pulls numeric outputs from earlier cells'
DuckDB views; **section cells above must have been run first**, because this cell intentionally avoids
recomputing heavy aggregations.


In [ ]:
# --- Synthesis ---
if not require_audit("Synthesis"):
    pass
else:
    findings = []

    # 1) Denylist candidates
    try:
        df = con.execute("""
            SELECT category, SUM(copy_pnl_usd) AS pnl, COUNT(*) AS n,
                   COUNT(DISTINCT cohort) AS cohorts_present,
                   COUNT(DISTINCT CASE WHEN copy_pnl_usd<0 THEN cohort END) AS cohorts_neg_signal
            FROM audit_x WHERE bt_window IN ('2025','2026') GROUP BY category
        """).df()
        cat_neg = df[(df["pnl"] < 0) & (df["n"] >= 30)].sort_values("pnl")
        if not cat_neg.empty:
            findings.append("**Categories to consider for denylist** (negative PnL, n≥30, primary windows):")
            for _, r in cat_neg.head(5).iterrows():
                findings.append(f"  - `{r['category']}` — pnl ${r['pnl']:,.0f} over {int(r['n'])} signals "
                                f"across {int(r['cohorts_present'])} cohort(s)")
        else:
            findings.append("**Denylist candidates**: none meet thresholds (n≥30, negative PnL).")
    except Exception as e:
        findings.append(f"_Denylist analysis skipped: {e}_")

    # 2) Top-K sizing recommendation from leader concentration
    try:
        df = con.execute("""
            WITH per_leader AS (
                SELECT cohort, resolution_bucket, sizing_rule, bt_window, leader_address,
                       SUM(copy_pnl_usd) AS pnl
                FROM audit_x WHERE bt_window IN ('2025','2026')
                GROUP BY 1,2,3,4,5
            ),
            ranked AS (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY cohort, resolution_bucket, sizing_rule, bt_window
                                              ORDER BY pnl DESC) AS rk,
                          COUNT(*)     OVER (PARTITION BY cohort, resolution_bucket, sizing_rule, bt_window) AS n_leaders
                FROM per_leader
                WHERE pnl > 0
            )
            SELECT cohort, resolution_bucket, sizing_rule, bt_window,
                   SUM(CASE WHEN rk <= 3 THEN pnl END) AS top3_pnl,
                   SUM(CASE WHEN rk <= 5 THEN pnl END) AS top5_pnl,
                   SUM(pnl)              AS total_positive_pnl,
                   AVG(n_leaders)        AS n_leaders
            FROM ranked GROUP BY 1,2,3,4
        """).df()
        if not df.empty:
            df["share_top3"] = df["top3_pnl"] / df["total_positive_pnl"].replace({0: np.nan})
            df["share_top5"] = df["top5_pnl"] / df["total_positive_pnl"].replace({0: np.nan})
            avg_top3 = df["share_top3"].mean()
            avg_top5 = df["share_top5"].mean()
            if pd.notna(avg_top3):
                if avg_top3 > 0.80:
                    rec = "reduce Top-K from 10 to 3"
                elif avg_top5 > 0.80:
                    rec = "reduce Top-K from 10 to 5"
                else:
                    rec = "keep Top-K=10; positive PnL is broadly distributed"
                findings.append(f"**Top-K recommendation**: {rec} "
                                f"(top-3 leaders capture {avg_top3*100:.0f}% of positive PnL on average; "
                                f"top-5: {avg_top5*100:.0f}%).")
    except Exception as e:
        findings.append(f"_Top-K analysis skipped: {e}_")

    # 3) Per-market cap recommendation
    try:
        # Re-derive peak exposures across primary combos (simple, all combos).
        df = con.execute(f"""
            WITH r AS (
                SELECT cohort, resolution_bucket, sizing_rule, bt_window, market_id,
                       trade_timestamp, resolution_date, copy_size_usd
                FROM audit_x WHERE bt_window IN ('2025','2026')
            ),
            pairs AS (
                SELECT a.cohort, a.resolution_bucket, a.sizing_rule, a.bt_window, a.market_id,
                       a.trade_timestamp AS t, SUM(b.copy_size_usd) AS exp_at_t
                FROM r a JOIN r b
                  ON a.cohort=b.cohort AND a.resolution_bucket=b.resolution_bucket
                 AND a.sizing_rule=b.sizing_rule AND a.bt_window=b.bt_window AND a.market_id=b.market_id
                 AND b.trade_timestamp <= a.trade_timestamp AND b.resolution_date > a.trade_timestamp
                GROUP BY 1,2,3,4,5,6
            )
            SELECT MAX(exp_at_t) / {STRATEGY_CAPITAL} AS peak_pct
            FROM pairs
            GROUP BY cohort, resolution_bucket, sizing_rule, bt_window, market_id
        """).df()
        if not df.empty:
            p95 = df["peak_pct"].quantile(0.95)
            rec_cap = max(0.03, np.ceil(p95 * 100) / 100)
            findings.append(f"**Per-market cap recommendation**: ~{rec_cap*100:.0f}% of strategy capital "
                            f"(95th pctile peak exposure across primary combos = {p95*100:.1f}%).")
    except Exception as e:
        findings.append(f"_Cap analysis skipped: {e}_")

    # 4) Slippage forensics
    try:
        df = con.execute("""
            SELECT
                SUM(CASE WHEN slippage_source='fallback' THEN 1 ELSE 0 END)*1.0/COUNT(*) AS fb_pct_overall,
                AVG(CASE WHEN slippage_source='fallback' THEN slippage_cents END) AS fb_avg,
                AVG(CASE WHEN slippage_source='next_fill' THEN slippage_cents END) AS nf_avg
            FROM audit_x WHERE bt_window IN ('2025','2026')
        """).df().iloc[0]
        fb_pct = float(df["fb_pct_overall"] or 0)
        flag = " — **HIGH**: slippage model is degraded; treat PnL with caution" if fb_pct > 0.30 else ""
        findings.append(f"**Slippage**: fallback share = {fb_pct*100:.1f}% across primary; "
                        f"avg next_fill = {df['nf_avg']:.2f}¢, avg fallback = {df['fb_avg']:.2f}¢.{flag}")
    except Exception as e:
        findings.append(f"_Slippage synthesis skipped: {e}_")

    # 5) Concrete iteration bullets
    iteration = []
    iteration.append("Strip the denylist categories (above) from the cohort copy universe before re-running.")
    iteration.append("Tighten Top-K per the recommendation above; concentration is the cheapest filter we have.")
    iteration.append("Add the per-market cap recommended above as a hard execution gate.")
    iteration.append("Re-run Stage 2 (slippage-window sensitivity) ONLY for combos with fallback_pct < 30%.")
    iteration.append("If a category-cohort combo is the source of >50% of PnL, build a dedicated diagnostic before relying on it.")
    findings.append("\n**What to change next iteration:**")
    findings.extend([f"- {b}" for b in iteration])

    print("\n".join(findings))
